In [83]:
import numpy as np
import pandas as pd
import os

folder_path = os.getcwd()
dataset_folder_path = os.path.join(folder_path,"..\\data_raw\\")
# Files inside dataset_raw
print(os.listdir(dataset_folder_path))


"""  Missing Value Summary
Products:
                            Missing Count  Missing %
product_category_name                 610   1.851234
product_description_lenght            610   1.851234
product_name_lenght                   610   1.851234
product_photos_qty                    610   1.851234
product_weight_g                        2   0.006070
product_height_cm                       2   0.006070
product_length_cm                       2   0.006070
product_width_cm                        2   0.006070

Orders:
                               Missing Count  Missing %
order_delivered_customer_date           2965   2.981668
order_delivered_carrier_date            1783   1.793023
order_approved_at                        160   0.160899

Order Reviews:
                        Missing Count  Missing %
review_comment_title       87656          88.341530
review_comment_message     58247          58.702532

"""

['customers_dataset.csv', 'geolocation_dataset.csv', 'orders_dataset.csv', 'order_items_dataset.csv', 'order_payments_dataset.csv', 'order_reviews_dataset.csv', 'products_dataset.csv', 'sellers_dataset.csv']


'  Missing Value Summary\nProducts:\n                            Missing Count  Missing %\nproduct_category_name                 610   1.851234\nproduct_description_lenght            610   1.851234\nproduct_name_lenght                   610   1.851234\nproduct_photos_qty                    610   1.851234\nproduct_weight_g                        2   0.006070\nproduct_height_cm                       2   0.006070\nproduct_length_cm                       2   0.006070\nproduct_width_cm                        2   0.006070\n\nOrders:\n                               Missing Count  Missing %\norder_delivered_customer_date           2965   2.981668\norder_delivered_carrier_date            1783   1.793023\norder_approved_at                        160   0.160899\n\nOrder Reviews:\n                        Missing Count  Missing %\nreview_comment_title       87656          88.341530\nreview_comment_message     58247          58.702532\n\n'

In [43]:
products_dataset = os.path.join(dataset_folder_path,'products_dataset.csv')
products_df = pd.read_csv(products_dataset)
products_df.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [44]:
missing_products = products_df[products_df.isna().any(axis=1)]
missing_products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [45]:
order_items_dataset = os.path.join(dataset_folder_path,'order_items_dataset.csv')
order_items_df = pd.read_csv(order_items_dataset)

order_items_df.dtypes

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

In [46]:
missing_products_categories = (products_df['product_category_name'].isna().sum() / products_df.shape[0])
missing_products_categories

np.float64(0.018512336499651)

In [47]:
# Since only 0.018% are missing it cna be dropped
products_df = products_df.dropna(subset=['product_category_name'])

In [48]:
missing_product_details = products_df[products_df.isna().any(axis=1)]
missing_product_details

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,baby,60.0,865.0,3.0,NaN,NaN,NaN,NaN


In [49]:
# Missing value imputation
baby_medians = (
    products_df[
        products_df['product_category_name'] == 'baby']
        [[
            'product_weight_g',
            'product_length_cm',
            'product_height_cm',
            'product_width_cm'
        ]].median()
)

for col in baby_medians.index:
    mask = (
        (products_df['product_category_name'] == 'baby')
        & (products_df[col].isna())
    )
    products_df.loc[mask, col] = baby_medians[col]

In [50]:
output_file_path = os.path.join(dataset_folder_path,"..\\data_cleaned\\products_dataset.csv")

products_df.to_csv(output_file_path, index=False)

###  Cleaning orders dataset

In [84]:
orders_dataset = os.path.join(dataset_folder_path,'orders_dataset.csv')
orders_df = pd.read_csv(orders_dataset)
orders_df.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [85]:
orders_df.dtypes
# We need to convert strings to timestamp for teh date columsn

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [86]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders_df[col] = pd.to_datetime(orders_df[col], errors='coerce')

In [87]:
#  For filling approval time we will use purchase time 
#  We will do purchase_time - approval_time and use its median to add to our payment timings to impute
median_approval_delay = (
    orders_df['order_approved_at'] - orders_df['order_purchase_timestamp']
).median()

orders_df.loc[orders_df['order_approved_at'].isna(),'order_approved_at'] = (orders_df.loc[orders_df['order_approved_at'].isna(), 'order_purchase_timestamp'] + median_approval_delay)

In [88]:
orders_df['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [89]:
missing_carrier_dates = orders_df[(orders_df['order_delivered_carrier_date'].isna()) & (orders_df['order_status'] == 'delivered')]
missing_carrier_dates


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23


In [90]:
# We will compute for NAN values whose delivery happened but dates are missing
median_ship_delay = (orders_df['order_delivered_carrier_date'] - orders_df['order_approved_at']).median()

mask = (orders_df['order_delivered_carrier_date'].isna()) & (orders_df['order_status'] == 'delivered')

orders_df.loc[mask, 'order_delivered_carrier_date'] = (orders_df.loc[mask, 'order_approved_at']+ median_ship_delay)
orders_df.loc[mask, 'order_delivered_customer_date'] = (orders_df.loc[mask, 'order_approved_at']+ median_ship_delay)

In [91]:
for status in ('shipped','canceled','unavailable','invoiced','processing','created','approved'):
    missing_carrier_dates = orders_df[(orders_df['order_delivered_carrier_date'].isna()) & (orders_df['order_status'] == status)]
    print(f"{status} is having  {missing_carrier_dates.shape[0]} missing order delivered carrier dates")

shipped is having  0 missing order delivered carrier dates
canceled is having  550 missing order delivered carrier dates
unavailable is having  609 missing order delivered carrier dates
invoiced is having  314 missing order delivered carrier dates
processing is having  301 missing order delivered carrier dates
created is having  5 missing order delivered carrier dates
approved is having  2 missing order delivered carrier dates


As we can see for **shipped** orders are having 0 missing values while for **cancelled**,**unavailable**,**invoiced** and **processing** it has missing values as they are not yet hence they are havign nan values

In [92]:
# Finding missing values based on statuses of customer delivery dates
for status in ('delivered','shipped','canceled','unavailable','invoiced','processing','created','approved'):
    missing_carrier_dates = orders_df[(orders_df['order_delivered_customer_date'].isna()) & (orders_df['order_status'] == status)]
    print(f"{status} is having  {missing_carrier_dates.shape[0]} missing order delivered customer dates")

delivered is having  7 missing order delivered customer dates
shipped is having  1107 missing order delivered customer dates
canceled is having  619 missing order delivered customer dates
unavailable is having  609 missing order delivered customer dates
invoiced is having  314 missing order delivered customer dates
processing is having  301 missing order delivered customer dates
created is having  5 missing order delivered customer dates
approved is having  2 missing order delivered customer dates


In [93]:
# Imputing missing values for the orders that were delivered but the customer delivery date is missing
orders_df[(orders_df['order_delivered_customer_date'].isna()) & (orders_df['order_status'] == 'delivered')]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [94]:
delivery_time = (orders_df['order_delivered_customer_date'] - orders_df['order_delivered_carrier_date'])
median_delivery_time = delivery_time.median()

mask = (orders_df['order_status'] == 'delivered') & (orders_df['order_delivered_customer_date'].isna())

orders_df.loc[mask, 'order_delivered_customer_date'] = (orders_df.loc[mask, 'order_delivered_carrier_date'] + median_delivery_time)

In [95]:
orders_df.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   0
order_delivered_carrier_date     1781
order_delivered_customer_date    2957
order_estimated_delivery_date       0
dtype: int64

In [ ]:
output_file_path = os.path.join(dataset_folder_path,"..\\data_cleaned\\orders_dataset.csv")

orders_df.to_csv(output_file_path, index=False)

In [97]:
orders_df.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [99]:
output_file_path = os.path.join(dataset_folder_path,"..\\data_cleaned\\orders_dataset.csv")
temp_df = pd.read_csv(output_file_path)
temp_df.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [66]:
"""
Order Reviews:
                        Missing Count       Missing %
review_comment_title       87656            88.341530
review_comment_message     58247            58.702532
"""

order_review_dataset = os.path.join(dataset_folder_path,"order_reviews_dataset.csv")
order_review_df = pd.read_csv(order_review_dataset)
order_review_df.columns

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='str')

In [67]:
order_review_df['has_comment_title'] = order_review_df['review_comment_title'].notna().astype(int)
order_review_df['has_comment_message'] = order_review_df['review_comment_message'].notna().astype(int)

order_review_df['has_comment_title'].value_counts()
# order_review_df['has_comment_message'].value_counts()

has_comment_title
0    87656
1    11568
Name: count, dtype: int64

### Working with duplicates for geolocation

In [68]:
geolocation_dataset = os.path.join(dataset_folder_path,'geolocation_dataset.csv')
geolocation_df = pd.read_csv(geolocation_dataset)

geolocation_df.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [69]:
geolocation_df.duplicated().sum() / geolocation_df.shape[0]

np.float64(0.2617883285024541)

In [70]:
# Since these duplicates are not adding any value we are going to remove them
geolocation_df = geolocation_df.drop_duplicates()

In [71]:
geolocation_df.duplicated().sum()

np.int64(0)

In [72]:
geolocation_df.dtypes

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

In [73]:
output_path = os.path.join(dataset_folder_path,"..\\data_cleaned","geolocation_dataset.csv")
geolocation_df.to_csv(output_path)

In [74]:
# Checking other datasets duplciates & data types
def data_profiling(file_path):
    df = pd.read_csv(file_path)

    print(f"Number of duplciate records:{df.duplicated().sum()}")
    print(f"Data type of columns:\n{df.dtypes}")


In [75]:
for file in ('customers_dataset.csv','order_items_dataset.csv','order_payments_dataset.csv','order_reviews_dataset.csv','sellers_dataset.csv'):
    file_path = os.path.join(dataset_folder_path,file)
    data_profiling(file_path)
    print()


Number of duplciate records:0
Data type of columns:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Number of duplciate records:0
Data type of columns:
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

Number of duplciate records:0
Data type of columns:
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

Number of duplciate records:0
Data type of columns:
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
re

In [76]:

# Converting shipping date to date time from string in orders_items table
order_items_dataset = os.path.join(dataset_folder_path,'order_items_dataset.csv')
order_items_df = pd.read_csv(order_items_dataset)

order_items_df['shipping_limit_date'] = pd.to_datetime(order_items_df['shipping_limit_date'])

In [77]:
order_items_df.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

In [78]:
output_file_path = os.path.join(dataset_folder_path,"..\\data_cleaned\\order_items_dataset.csv")
order_items_df.to_csv(output_file_path)

In [79]:
orders_dataset = os.path.join(dataset_folder_path,"..\\data_cleaned\\orders_dataset.csv")
orders_df =  pd.read_csv(orders_dataset)
orders_df.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object